In [36]:
"""
Reddit Comment Labeling Pipeline (LM Studio, OpenAI-compatible API)
-------------------------------------------------------------------
- Input: dictionaries (Reddit-style with keys 'id' and 'body'), OR a dict mapping id->text,
         OR a pandas DataFrame with columns ['id','body'].
- Retries: Up to 3 attempts per (comment, task) with exponential backoff.
- Strict JSON validation against task schemas; otherwise retry.
- Output: NDJSON (one line per (comment_id, task)).
- Tasks implemented:
    - stance_intensity
    - epistemic_modality
    - justification_density
    - responsiveness
    - agreement
    - civility
    - sarcasm

Extensions:
    - For each comment we additionally run an argument extraction call:
        {
          "task": "argument_extraction",
          "arguments": [ "<string>", ... ],
          "confidence": <float in [0,1]>
        }
      The extracted arguments + confidence are added to each NDJSON line:
        "arguments": [...],
        "arguments_confidence": <float or null>,
        "arguments_error": <string or null>

    - Parent-ID of the comment is stored in the NDJSON output as "parent_id"
      (string or null). For DataFrame input this uses `parent_col` if available.
      For the generic run_pipeline (no DF), parent_id is always null.

Requirements:
    pip install requests pandas
    reddit data has to be stored locally in a folder called 'data' (optional demo below).

LM Studio:
    - Enable the local HTTP server in LM Studio (OpenAI-compatible API).
    - Default endpoint: http://localhost:1234/v1
    - Set MODEL_NAME to the exact local model identifier shown in LM Studio.
"""

import json
import time
import os
from typing import Dict, Any, List, Optional, Iterable, Tuple, Union, Set, Callable
from pathlib import Path

import pandas as pd
import requests
import numpy as np

from openai import OpenAI



In [37]:
# ------------------------
# Configuration
# ------------------------

LMSTUDIO_BASE_URL = "http://127.0.0.1:1234"  # Change if LM Studio runs on a different host/port
MODEL_NAME = "ibm/granite-3.2-8b"            # e.g., "qwen2.5-7b-instruct"
TIMEOUT_SECONDS = 60                         # HTTP request timeout
MAX_RETRIES = 3                              # Max attempts per (comment, task)
RETRY_BACKOFF_SECONDS = 1.5                  # Exponential backoff base

KIT_base_url = "https://ki-toolbox.scc.kit.edu/api/v1"
# KIT_api_key = os.environ["KIT_AI_API_KEY"]
KIT_model = 'kit.gpt-oss-120b'

# HoreKa vLLM Server (via SSH-Tunnel)
HOREKA_BASE_URL = "http://localhost:8000"
HOREKA_MODEL = "/hkfs/work/workspace/scratch/unoim-llm_models/hf_cache/models/meta-llama/Llama-3.1-70B-Instruct"
# %%
# ------------------------
# Read reddit data (robust)
# ------------------------

p_com = Path("../data/conversation_threads_flat.ndjson")

print("Exists comments:", p_com.exists(), p_com.resolve())

required = ["comment_id", "body", "parent_id", "submission_id", "user", "created_utc", "depth"]

if p_com.exists():
    records = []
    skipped_lines = []

    with open(p_com, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                skipped_lines.append((i, str(e), line[:200]))

    if skipped_lines:
        print(f"⚠️  {len(skipped_lines)} kaputte Zeile(n) übersprungen:")
        for lineno, err, preview in skipped_lines:
            print(f"  Zeile {lineno}: {err}")
            print(f"  Preview: {preview}")

    df_comments = pd.DataFrame(records)
    print(f"✅ Geladen: {len(df_comments)} Zeilen, übersprungen: {len(skipped_lines)}")

else:
    df_comments = pd.DataFrame()
    print("⚠️  Datei nicht gefunden, leerer DataFrame erstellt.")

if not df_comments.empty:
    missing = [c for c in required if c not in df_comments.columns]
    if missing:
        raise ValueError(f"Missing required columns in df_comments: {missing}")

    df_comments["comment_id"]    = df_comments["comment_id"].astype("string")
    df_comments["parent_id"]     = df_comments["parent_id"].astype("string")
    df_comments["submission_id"] = df_comments["submission_id"].astype("string")
    df_comments["user"]          = df_comments["user"].astype("string")
    df_comments["body"]          = df_comments["body"].astype("string")
else:
    df_comments = pd.DataFrame(columns=required)

print(df_comments.head())
print("Columns:", list(df_comments.columns))


Exists comments: True /Users/arthur/DataspellProjects/reddit-l/data/conversation_threads_flat.ndjson
⚠️  1 kaputte Zeile(n) übersprungen:
  Zeile 2599: Unterminated string starting at: line 1 column 462 (char 461)
  Preview: {"thread_id": "1di4uk5", "submission_id": "1di4uk5", "root_id": "l91jfzl", "comment_id": "l93uubq", "parent_id": "l92wv7b", "link_id": "1di4uk5", "body": "Yeah it’s just causing some Ukrainian lives. 
✅ Geladen: 2598 Zeilen, übersprungen: 1
  thread_id submission_id  root_id comment_id parent_id  link_id  \
0   1do7blq       1do7blq  la7kwyu    la7kwyu   1do7blq  1do7blq   
1   1do7blq       1do7blq  la7mg5r    la7mg5r   1do7blq  1do7blq   
2   1do7blq       1do7blq  la7mg5r    la7twx3   la7mg5r  1do7blq   
3   1do7blq       1do7blq  la7mg5r    la83okv   la7twx3  1do7blq   
4   1do7blq       1do7blq  la7mg5r    la8b9n6   la83okv  1do7blq   

                                                body             user  score  \
0  \nRemember that TrueReddit is a place to e

In [38]:
print(df_comments.head())

  thread_id submission_id  root_id comment_id parent_id  link_id  \
0   1do7blq       1do7blq  la7kwyu    la7kwyu   1do7blq  1do7blq   
1   1do7blq       1do7blq  la7mg5r    la7mg5r   1do7blq  1do7blq   
2   1do7blq       1do7blq  la7mg5r    la7twx3   la7mg5r  1do7blq   
3   1do7blq       1do7blq  la7mg5r    la83okv   la7twx3  1do7blq   
4   1do7blq       1do7blq  la7mg5r    la8b9n6   la83okv  1do7blq   

                                                body             user  score  \
0  \nRemember that TrueReddit is a place to engag...    AutoModerator      1   
1  "nutrition influencer" is a weird way of sayin...     TheShipEliza     24   
2  "Liar" really undersells it. "Grifter" works b...  wholetyouinhere     25   
3  Was gonna say charlatan initially but felt lik...     TheShipEliza      7   
4  I think that's the perfect term. But it doesn'...  wholetyouinhere      6   

   created_utc  ... depth                                   submission_title  \
0   1719325626  ...     0  Nut

In [39]:
# ------------------------
# Task-Spezifikation
# ------------------------

TASK_SPECS: Dict[str, Dict[str, Any]] = {
    "stance_intensity": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "stance_intensity",
        "score_key": "score",
        "score_type": (int, float),   # numerisch, außer wenn "ABSTAIN"
        "confidence_range": (0.0, 1.0),
        "score_range": (1, 6),
        "allow_abstain": True,
        "label_key": None,
    },
    "epistemic_modality": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "epistemic_modality",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, 1.0),
        "allow_abstain": True,
        "label_key": None,
    },
    "justification_density": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "justification_density",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, float("inf")),  # nichtnegativ
        "allow_abstain": True,
        "label_key": None,
    },
    "responsiveness": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "responsiveness",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, 1.0),
        "allow_abstain": True,
        "label_key": None,
    },
    "agreement": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "agreement",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (-1.0, 1.0),
        "allow_abstain": True,
        "label_key": None,
    },
    "civility": {
        "schema_keys": {"task", "label", "confidence"},
        "task_value": "civility",
        "score_key": None,
        "label_key": "label",
        "label_type": (int,),         # int 1..6, außer wenn "ABSTAIN"
        "score_range": (1.0, 6.0),
        "confidence_range": (0.0, 1.0),
        "allow_abstain": True,
    },
    "sarcasm": {
        "schema_keys": {"task", "label", "confidence"},
        "task_value": "sarcasm",
        "score_key": None,
        "label_key": "label",
        "label_type": (int,),         # 0 oder 1, außer wenn "ABSTAIN"
        "score_range": (0.0, 1.0),    # 0 = kein Sarkasmus, 1 = Sarkasmus
        "confidence_range": (0.0, 1.0),
        "allow_abstain": True,
    },
}


In [40]:
# ------------------------
# HTTP call utilities
# ------------------------

def call_lmstudio_chat(messages: List[Dict[str, str]], temperature: float = 0.0) -> str:
    """
    Call LM Studio (OpenAI-compatible) Chat Completions API and return raw text.

    Raises:
        requests.RequestException on network/HTTP errors.
    """
    url = f"{LMSTUDIO_BASE_URL}/v1/chat/completions"
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": temperature,
        "stream": False,
    }
    resp = requests.post(url, json=payload, timeout=TIMEOUT_SECONDS)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"].strip()


# ------------------------
# HTTP call utilities
# ------------------------

def call_kit_chat(messages: List[Dict[str, str]], temperature: float = 0.0) -> str:
    """
    Call LM Studio (OpenAI-compatible) Chat Completions API and return raw text.

    Raises:
        requests.RequestException on network/HTTP errors.
    """
    url = f"{KIT_base_url}/v1/chat/completions"
    payload = {
        "model": KIT_model,
        "messages": messages,
        "temperature": temperature,
        "stream": False,
    }
    #resp = requests.post(url,api_key = KIT_api_key, json=payload, timeout=TIMEOUT_SECONDS)

    client = OpenAI(base_url=KIT_base_url)
    try:
        # Send chat request
        resp = client.chat.completions.create(
            model=KIT_model,
            messages=messages,
            temperature=temperature)
    except KeyError as exc:
        raise RuntimeError("Error") from exc

    #resp.raise_for_status()
    #data = resp.json()
    return resp.choices[0].message.content.strip()

def call_horeka_chat(messages: List[Dict[str, str]], temperature: float = 0.0) -> str:
    client = OpenAI(
        base_url=f"{HOREKA_BASE_URL}/v1",
        api_key="dummy"  # vLLM braucht keinen echten Key
    )
    resp = client.chat.completions.create(
        model=HOREKA_MODEL,
        messages=messages,
        temperature=temperature
    )
    return resp.choices[0].message.content.strip()

In [41]:



def _is_number(x: Any) -> bool:
    """Helper: check if value is an int or float."""
    return isinstance(x, (int, float))


# ------------------------
# Response validation (label tasks)
# ------------------------

def validate_response(task: str, obj: Dict[str, Any]) -> Optional[str]:
    """
    Validate a single task response against TASK_SPECS[task].

    Supports:
    - "ABSTAIN" for score/label if allow_abstain=True

    Returns:
        None if valid, otherwise a string describing the error.
    """
    spec = TASK_SPECS[task]

    if not isinstance(obj, dict):
        return "Response is not a JSON object"

    missing = spec["schema_keys"] - set(obj.keys())
    if missing:
        return f"Missing required keys: {sorted(missing)}"

    if obj.get("task") != spec["task_value"]:
        return f'Field "task" must be "{spec["task_value"]}"'

    # Validate confidence
    conf = obj.get("confidence")
    if not _is_number(conf):
        return '"confidence" must be a number'
    lo_c, hi_c = spec["confidence_range"]
    if not (lo_c <= conf <= hi_c):
        return f'"confidence" must be in [{lo_c}, {hi_c}]'

    label_key = spec.get("label_key")
    score_key = spec.get("score_key")

    # Label-based tasks (e.g. civility, sarcasm)
    if label_key is not None:
        val = obj.get(label_key)

        # Handle ABSTAIN
        if isinstance(val, str):
            if spec.get("allow_abstain") and val == "ABSTAIN":
                return None
            return f'"{label_key}" must be an integer in range or "ABSTAIN"'

        label_type = spec.get("label_type", (int,))
        if not isinstance(val, label_type):
            return f'"{label_key}" has wrong type (expected {label_type})'

        lo, hi = spec["score_range"]
        if not (lo <= float(val) <= hi):
            return f'"{label_key}" out of range [{lo}, {hi}]'
        return None

    # Score-based tasks
    if score_key is not None:
        val = obj.get(score_key)

        if isinstance(val, str):
            if spec.get("allow_abstain") and val == "ABSTAIN":
                return None
            return f'"{score_key}" must be a number or "ABSTAIN"'

        if not _is_number(val):
            return f'"{score_key}" must be a number'
        lo, hi = spec["score_range"]
        if not (lo <= float(val) <= hi):
            return f'"{score_key}" out of range [{lo}, {hi}]'
        return None

    return None


# ------------------------
# Validation: Argument extraction
# ------------------------

def validate_arguments(obj: Dict[str, Any]) -> Optional[str]:
    """
    Validate the argument_extraction response.

    Expected schema:
    {
      "task": "argument_extraction",
      "arguments": [
        { "claim": "<string>", "proof": "<string>" },
        ...
      ],
      "confidence": <float in [0,1]>
    }

    Returns:
        None if valid, otherwise a string describing the error.
    """
    if not isinstance(obj, dict):
        return "Response is not a JSON object"

    if obj.get("task") != "argument_extraction":
        return 'Field "task" must be "argument_extraction"'

    if "arguments" not in obj or "confidence" not in obj:
        return "Missing required keys: 'arguments' and/or 'confidence'"

    args = obj["arguments"]
    if not isinstance(args, list):
        return '"arguments" must be a list'

    for i, a in enumerate(args):
        if not isinstance(a, dict):
            return f'"arguments[{i}]" must be an object with keys "claim" and "proof"'
        if "claim" not in a or "proof" not in a:
            return f'"arguments[{i}]" must contain keys "claim" and "proof"'
        if not isinstance(a["claim"], str):
            return f'"arguments[{i}].claim" must be a string'
        if not isinstance(a["proof"], str):
            return f'"arguments[{i}].proof" must be a string'

    conf = obj["confidence"]
    if not _is_number(conf):
        return '"confidence" must be a number'
    if not (0.0 <= conf <= 1.0):
        return '"confidence" must be in [0,1]'

    return None


# ------------------------
# Prompt construction
# ------------------------

def build_user_prompt(task: str, text: str, parent_text: Optional[str] = None) -> str:
    """
    Compose the user message for labeling tasks that includes:
      - TEXT
      - optional PARENT_TEXT
      - a directive specifying which task to perform.
    """
    parts: Dict[str, Any] = {"TEXT": text}
    if parent_text is not None and str(parent_text).strip():
        parts["PARENT_TEXT"] = str(parent_text)
    directive = TASK_INSTRUCTION_TEMPLATE.format(task_name=task)
    return json.dumps(parts, ensure_ascii=False) + "\n\n" + directive


def build_argument_prompt(text: str, parent_text: Optional[str] = None) -> str:
    """
    Compose the user message for argument_extraction.
    """
    pt = parent_text if (parent_text is not None and str(parent_text).strip()) else "N/A"
    return ARG_USER_TEMPLATE.format(text=text, parent_text=pt)


# ------------------------
# Single annotation with retries (label tasks)
# ------------------------

def annotate_one(task: str, text: str, parent_text: Optional[str] = None) -> Dict[str, Any]:
    """
    Run a single (task, text[, parent_text]) through LM Studio with retries
    and strict JSON validation.

    Returns:
        - On success: the parsed JSON object produced by the model.
        - On failure: {"task": task, "error": "..."} after all retries fail.
    """
    last_err: Optional[str] = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_user_prompt(task, text, parent_text)},
            ]
            #raw = call_lmstudio_chat(messages, temperature=0.0)
            raw = call_horeka_chat(messages, temperature=0.0)

            obj = json.loads(raw)
            err = validate_response(task, obj)
            if err is None:
                return obj
            else:
                last_err = f"Schema validation failed (attempt {attempt}): {err}"

        except requests.RequestException as e:
            last_err = f"HTTP error (attempt {attempt}): {e}"

        except json.JSONDecodeError as e:
            last_err = f"JSON parse error (attempt {attempt}): {e}"

        # Exponential backoff
        time.sleep((RETRY_BACKOFF_SECONDS ** attempt))

    return {
        "task": task,
        "error": last_err or "Unknown error",
    }


def extract_arguments(text: str, parent_text: Optional[str] = None) -> Dict[str, Any]:
    """
    Run argument_extraction for a given text (and optional parent_text) with retries
    and strict JSON validation.

    Returns:
        On success:
        {
          "task": "argument_extraction",
          "arguments": [
            { "claim": "<string>", "proof": "<string>" },
            ...
          ],
          "confidence": float
        }

        On failure:
        {
          "task": "argument_extraction",
          "error": "..."
        }
    """
    last_err: Optional[str] = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            messages = [
                {"role": "system", "content": ARG_SYSTEM_PROMPT},
                {"role": "user", "content": build_argument_prompt(text, parent_text)},
            ]
            #raw = call_lmstudio_chat(messages, temperature=0.0)
            raw = call_horeka_chat(messages, temperature=0.0)
            obj = json.loads(raw)
            err = validate_arguments(obj)
            if err is None:
                return obj
            else:
                last_err = f"Argument schema validation failed (attempt {attempt}): {err}"
        except requests.RequestException as e:
            last_err = f"HTTP error (attempt {attempt}): {e}"
        except json.JSONDecodeError as e:
            last_err = f"JSON parse error (attempt {attempt}): {e}"

        # Exponential backoff
        time.sleep((RETRY_BACKOFF_SECONDS ** attempt))

    return {
        "task": "argument_extraction",
        "error": last_err or "Unknown error",
    }


# ------------------------
# NDJSON writer
# ------------------------

def write_ndjson_line(fp, obj: Dict[str, Any]) -> None:
    """
    Write a single JSON object as one NDJSON line to an open file handle.
    """
    fp.write(json.dumps(obj, ensure_ascii=False) + "\n")


# ------------------------
# Input normalization for generic pipeline
# ------------------------

NormalizedItem = Tuple[str, str]  # (comment_id, body_text)


def _normalize_input(
    comments: Union[
        Dict[str, Any],       # single Reddit dict with keys 'id' & 'body' OR mapping id->text
        List[Dict[str, Any]], # list of Reddit-style dicts
        pd.DataFrame,         # DataFrame with columns ['id','body'] or ['comment_id','body']
    ]
) -> Iterable[NormalizedItem]:
    """
    Normalize different input shapes to an iterator of (comment_id, body_text).

    Accepted forms:
      - Single Reddit-style dict with 'id' and 'body'
      - List of Reddit-style dicts with 'id' and 'body'
      - Mapping {id: text} (dict)
      - pd.DataFrame with columns ['id','body'] or ['comment_id','body']
    """
    if isinstance(comments, pd.DataFrame):
        # Prefer 'comment_id' if present, otherwise 'id'
        if "comment_id" in comments.columns:
            id_col = "comment_id"
        elif "id" in comments.columns:
            id_col = "id"
        else:
            raise ValueError("DataFrame input must contain column 'comment_id' or 'id'.")
        if "body" not in comments.columns:
            raise ValueError("DataFrame input must contain column 'body'.")

        for _, row in comments.iterrows():
            cid = str(row[id_col])
            text = str(row["body"])
            if text and cid:
                yield (cid, text)
        return

    if isinstance(comments, dict):
        # Case 1: single Reddit dict with 'id' and 'body'
        if "id" in comments and "body" in comments:
            yield (str(comments["id"]), str(comments["body"]))
            return
        # Case 2: mapping {id: text}
        for k, v in comments.items():
            cid = str(k)
            text = str(v)
            if text and cid:
                yield (cid, text)
        return

    if isinstance(comments, list):
        # List of Reddit-style dicts
        for item in comments:
            if not isinstance(item, dict):
                raise ValueError("List input must contain dictionaries with keys ['id','body'].")
            if "id" not in item or "body" not in item:
                raise ValueError("Each dictionary in the list must have 'id' and 'body'.")
            yield (str(item["id"]), str(item["body"]))
        return

    raise TypeError(
        "Unsupported input type. Provide a DataFrame with ['comment_id','body'] or ['id','body'], "
        "a dict mapping id->text, a single Reddit dict with 'id'/'body', or a list of such dicts."
    )


# ------------------------
# Generic main pipeline (no parent lookup)
# ------------------------

def run_pipeline(
    comments: Union[Dict[str, Any], List[Dict[str, Any]], pd.DataFrame],
    tasks: Optional[List[str]] = None,
    ndjson_path: str = "labels.ndjson",
) -> None:
    """
    Run the labeling pipeline on generic input (without parent-text lookup).

    Output format (NDJSON):
        One line per (comment_id, task), with JSON object:
        {
          "comment_id": "<id string>",
          "body": "<comment text>",
          "submission_id": null,
          "created_utc": null,
          "user": null,
          "parent_id": null,
          "depth": null,
          "comment_index": <int>,
          "task": "<task_name>",
          "result": { ... },
          "arguments": [...],
          "arguments_confidence": <float or null>,
          "arguments_error": <string or null>
        }
    """
    if tasks is None:
        tasks = list(TASK_SPECS.keys())

    iterator = list(_normalize_input(comments))

    with open(ndjson_path, "w", encoding="utf-8") as f:
        for idx, (comment_id, text) in enumerate(iterator):
            # Argument extraction once per comment
            arg_res = extract_arguments(text, parent_text=None)
            if "error" in arg_res:
                arguments = []
                arg_conf = None
                arg_err = arg_res["error"]
            else:
                arguments = arg_res.get("arguments", [])
                arg_conf = arg_res.get("confidence", None)
                arg_err = None

            # Run all labeling tasks for this comment
            for task in tasks:
                result = annotate_one(task, text, parent_text=None)
                out = {
                    # Meta information (as far as available in this generic case)
                    "comment_id": comment_id,
                    "body": text,
                    "submission_id": None,
                    "created_utc": None,
                    "user": None,
                    "parent_id": None,
                    "depth": None,

                    # Pipeline / label information
                    "comment_index": idx,
                    "task": task,
                    "result": result,
                    "arguments": arguments,
                    "arguments_confidence": arg_conf,
                    "arguments_error": arg_err,
                }
                write_ndjson_line(f, out)


# ------------------------
# Resume helper: already labeled (comment_id, task) pairs
# ------------------------

def _load_done_pairs(ndjson_path: str) -> Set[Tuple[str, str]]:
    """
    Read (comment_id, task) pairs from an existing NDJSON file in order to
    avoid duplicated work (resume functionality).

    Returns:
        A set of (comment_id, task) tuples that have already been processed.
    """
    done: Set[Tuple[str, str]] = set()
    if not os.path.exists(ndjson_path):
        return done
    with open(ndjson_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                cid = str(obj.get("comment_id", ""))
                t = str(obj.get("task", ""))
                if cid and t:
                    done.add((cid, t))
            except Exception:
                # Ignore malformed lines
                continue
    return done


# ------------------------
# DataFrame-based pipeline with parent text & parent ID
# ------------------------

def _jsonable(v: Any) -> Any:
    """Convert pandas/numpy scalars to plain Python types for JSON."""
    if v is None:
        return None
    # pd.isna funktioniert auch für viele numpy/pandas types
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    if isinstance(v, (np.integer, np.floating)):
        return v.item()
    return v

def _make_abstain_result(task: str, reason: str = "EMPTY_BODY") -> Dict[str, Any]:
    """Create a schema-like ABSTAIN result without calling the model."""
    # score-tasks vs label-tasks
    if task in {"civility", "sarcasm"}:
        return {"task": task, "label": "ABSTAIN", "confidence": 0.0, "note": reason}
    else:
        return {"task": task, "score": "ABSTAIN", "confidence": 0.0, "note": reason}

def label_dataframe(
        df: pd.DataFrame,
        tasks: Optional[List[str]] = None,
        ndjson_path: str = "labels.ndjson",
        id_col: str = "comment_id",
        text_col: str = "body",
        batch_size: int = 500,
        start_index: int = 0,
        end_index: Optional[int] = None,
        skip_existing: bool = True,
        progress: Optional[Callable[[int, int, Dict[str, Any]], None]] = None,
        parent_col: str = "parent_id",
) -> Dict[str, Any]:

    assert id_col in df.columns, f"Column '{id_col}' is missing in the DataFrame."
    assert text_col in df.columns, f"Column '{text_col}' is missing in the DataFrame."

    if tasks is None:
        tasks = list(TASK_SPECS.keys())

    n_total = len(df)
    if end_index is None or end_index > n_total:
        end_index = n_total
    if start_index < 0:
        start_index = 0
    if start_index >= end_index:
        return {
            "processed_rows": 0,
            "written_records": 0,
            "skipped_records": 0,
            "errors": 0,
            "note": "Nothing to process (start_index >= end_index).",
        }

    # parent lookup: comment_id -> body (für Reply-zu-Comment Fälle)
    parent_lookup = dict(
        zip(df[id_col].astype("string").tolist(), df[text_col].astype("string").tolist())
    ) if parent_col in df.columns else {}

    already_done: Set[Tuple[str, str]] = set()
    if skip_existing:
        already_done = _load_done_pairs(ndjson_path)

    written_records = 0
    skipped_records = 0
    error_count = 0
    processed_rows = 0

    with open(ndjson_path, "a", encoding="utf-8") as fp:
        for batch_start in range(start_index, end_index, batch_size):
            batch_end = min(batch_start + batch_size, end_index)
            batch = df.iloc[batch_start:batch_end]

            for local_idx, row in batch.iterrows():
                comment_id = str(row[id_col]) if pd.notna(row[id_col]) else ""
                text = str(row[text_col]) if pd.notna(row[text_col]) else ""

                # Meta: ALLE anderen Spalten mitschreiben
                meta = {}
                for c in df.columns:
                    if c in {id_col, text_col}:
                        continue
                    meta[c] = _jsonable(row[c]) if c in row.index else None

                # Parent ID
                parent_id = None
                if parent_col in df.columns and pd.notna(row.get(parent_col, None)):
                    parent_id = str(row[parent_col])

                # Parent-Text:
                # 1) Wenn parent_id ein Kommentar ist -> parent_lookup
                # 2) Sonst (typisch depth==0): Submission-Kontext aus submission_title/selftext
                parent_text = None
                if parent_id:
                    if parent_id in parent_lookup:
                        parent_text = parent_lookup[parent_id]
                    else:
                        # Submission-Kontext
                        title = row.get("submission_title", None)
                        selftext = row.get("submission_selftext", None)
                        title = "" if (title is None or (isinstance(title, float) and np.isnan(title))) else str(title)
                        selftext = "" if (selftext is None or (isinstance(selftext, float) and np.isnan(selftext))) else str(selftext)
                        combined = (title.strip() + "\n\n" + selftext.strip()).strip()
                        parent_text = combined if combined else None

                # Für leere Texte: keine LLM Calls; trotzdem "mitnehmen" (je Task ABSTAIN schreiben)
                if (not comment_id) or (not text) or (not text.strip()):
                    # Argumente leer
                    arguments, arg_conf, arg_err = [], None, "EMPTY_BODY"
                    for task in tasks:
                        if skip_existing and (comment_id, task) in already_done:
                            skipped_records += 1
                            continue
                        out = {
                            "comment_id": comment_id,
                            "body": text,
                            "parent_id": parent_id,
                            "comment_index": int(local_idx),
                            "task": task,
                            "result": _make_abstain_result(task, reason="EMPTY_BODY"),
                            "arguments": arguments,
                            "arguments_confidence": arg_conf,
                            "arguments_error": arg_err,
                            "meta": meta,
                        }
                        write_ndjson_line(fp, out)
                        written_records += 1
                        if written_records % 100 == 0:
                            fp.flush()

                    processed_rows += 1
                    if progress:
                        progress(processed_rows, end_index - start_index, {
                            "written_records": written_records,
                            "skipped_records": skipped_records,
                            "errors": error_count
                        })
                    continue

                # Argument extraction einmal pro Kommentar
                arg_res = extract_arguments(text, parent_text=parent_text)
                if "error" in arg_res:
                    arguments = []
                    arg_conf = None
                    arg_err = arg_res["error"]
                else:
                    arguments = arg_res.get("arguments", [])
                    arg_conf = arg_res.get("confidence", None)
                    arg_err = None

                # Tasks laufen lassen
                for task in tasks:
                    if skip_existing and (comment_id, task) in already_done:
                        skipped_records += 1
                        continue

                    try:
                        result = annotate_one(task, text, parent_text=parent_text)

                        out = {
                            "comment_id": comment_id,
                            "body": text,
                            "parent_id": parent_id,
                            "comment_index": int(local_idx),
                            "task": task,
                            "result": result,
                            "arguments": arguments,
                            "arguments_confidence": arg_conf,
                            "arguments_error": arg_err,
                            "meta": meta,  # <- hier ist jetzt wirklich alles drin
                        }
                        write_ndjson_line(fp, out)
                        written_records += 1
                        if written_records % 100 == 0:
                            fp.flush()
                    except Exception:
                        error_count += 1

                processed_rows += 1

                if progress:
                    progress(processed_rows, end_index - start_index, {
                        "written_records": written_records,
                        "skipped_records": skipped_records,
                        "errors": error_count
                    })

    return {
        "processed_rows": processed_rows,
        "written_records": written_records,
        "skipped_records": skipped_records,
        "errors": error_count,
    }


In [42]:
# ------------------------
# System prompt (labeling tasks)
# ------------------------

SYSTEM_PROMPT = """You are a careful, literal discourse annotator. Use ONLY the provided TEXT,
PARENT_TEXT (if given), and TOPIC_DEF. Do not infer beyond the text or use any external knowledge.
Output must be strictly valid JSON matching the registered schema for the task. Include a "confidence" field in [0,1].

You MUST prefer returning "ABSTAIN" over guessing. If the evidence for a numeric/ordinal label is weak,
conflicting, or the text is very short, return "ABSTAIN" for that task.

English only. If TEXT has fewer than ~15 tokens, or evidence is weak/ambiguous, return "ABSTAIN"
instead of trying to infer a score/label. When you return "ABSTAIN", still provide a confidence score
in [0,1] reflecting your confidence that abstaining is appropriate (typically low, e.g., <= 0.4).

B.2 Instruction templates and schemas
All tasks share the same idea: return a single JSON object for the requested task.

IMPORTANT:
- For each task, the main numeric/ordinal field ("score" or "label") can either be:
  - a valid number in the specified range, OR
  - the string "ABSTAIN" (if you cannot reliably decide).
- Never output any other strings in place of numeric scores/labels.

--------------------------------------------------
Stance intensity (ordinal; [1,6])
--------------------------------------------------
TASK: Extract the topic of this comment and then based on the topic the author's stance
on a 1–6 scale on this topic (1 = strongly against, 3–4 = neutral/unclear, 6 = strongly in favor).

Use "ABSTAIN" if:
- there is no clear evidence of stance toward the topic, or
- the text is very short or off-topic, or
- signals are too mixed/ambiguous to place on the scale.

JSON schema:
{
  "task": "stance_intensity",
  "score": <int in [1, 6] OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Epistemic modality (continuous; [0,1])
--------------------------------------------------
TASK: Score epistemic modality: density of hedges, modal verbs, and explicit uncertainty,
normalized by length. Higher = more expressed humility/uncertainty.

If you cannot reliably judge (e.g., very short or unclear text), return "ABSTAIN".

JSON schema:
{
  "task": "epistemic_modality",
  "score": <float in [0.0,1.0] OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Justification density (per 100 words)
--------------------------------------------------
TASK: Count distinct justification units (claim + warrant), normalized per 100 words.
If uncertain, or if there is not enough content to identify justification units, return "ABSTAIN".

JSON schema:
{
  "task": "justification_density",
  "score": <non-negative float OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Responsiveness (continuous; [0,1])
--------------------------------------------------
TASK: Rate how directly this reply addresses its {PARENT_TEXT} (semantic overlap/engagement).

If PARENT_TEXT is not provided, or if the relation between TEXT and PARENT_TEXT is unclear,
return "ABSTAIN".

JSON schema:
{
  "task": "responsiveness",
  "score": <float in [0.0,1.0] OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Agreement (continuous; [-1,1])
--------------------------------------------------
TASK: Rate agreement with {PARENT_TEXT}.
-1 = contradicts, 0 = neutral/unrelated, 1 = fully agrees.

If PARENT_TEXT is not provided, or if agreement cannot be reliably determined (e.g., off-topic,
sarcastic or ambiguous content without clear polarity), return "ABSTAIN".

JSON schema:
{
  "task": "agreement",
  "score": <float in [-1.0,1.0] OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Civility (ordinal 1–6)
--------------------------------------------------
TASK: Rate civility on 1 (highly uncivil/insulting) to 6 (highly civil).

If civility is hard to judge (e.g., context missing, mixed cues, or text too short),
return "ABSTAIN".

JSON schema:
{
  "task": "civility",
  "label": <integer 1..6 OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Sarcasm (binary 0/1)
--------------------------------------------------
TASK: Detect whether the TEXT is sarcastic or clearly ironic regarding its main topic.

- Extract the topic of this comment.
- Use 1 if there is clear sarcastic or ironic intent (e.g., praise used to convey criticism,
  exaggerated contrast between words and obvious reality, well-known sarcastic formulae).
- Use 0 if the text is clearly non-sarcastic and literal.
- If signals are ambiguous, context is insufficient, or the text is too short to decide,
  return "ABSTAIN".

JSON schema:
{
  "task": "sarcasm",
  "label": <0 or 1 OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}
"""

# Template that tells the model which single task to execute now
TASK_INSTRUCTION_TEMPLATE = (
    "Now perform ONLY the task = {task_name}. "
    "Return strictly valid JSON for that task and nothing else (no Markdown). "
    "Ensure keys and value ranges match the schema exactly."
)

# ------------------------
# Argument-Extraktion Prompt
# ------------------------

ARG_SYSTEM_PROMPT = """
You are an argument mining annotator.
Use ONLY the provided TEXT and PARENT_TEXT (if given). Do not infer beyond the text.

Your task: extract explicit argumentative units from TEXT.
Each argument must consist of:
- a 'claim' (the central assertion), and
- a 'proof' (the supporting reason/evidence),
both as short strings.

Output strictly valid JSON:
{
  "task": "argument_extraction",
  "arguments": [
      { "claim": "<string>", "proof": "<string>" },
      { "claim": "<string>", "proof": "<string>" }
  ],
  "confidence": <float in [0,1]>
}

Guidelines:
- "arguments" must be a list of objects, each with keys "claim" and "proof".
- If TEXT contains no clear arguments, use an empty list [].
- Claims and proofs must be grounded in the text; do NOT hallucinate.
- Claims and proofs should be concise summaries or minimal spans.
- Prefer under-detection (few arguments) over over-detection.
"""


ARG_USER_TEMPLATE = (
    "TEXT: {text}\n\n"
    "If available, PARENT_TEXT (context for replies): {parent_text}\n\n"
    "Now perform ONLY argument_extraction as described in the system prompt. "
    "Return strictly valid JSON and nothing else."
)


In [58]:
# Demo-Aufruf
if not df_comments.empty:
    n_entries = 40000   # <== adapt as needed (number of comments to label)

    tasks = [
        "stance_intensity",
        "civility",
        "epistemic_modality",
        "justification_density",
        "responsiveness",
        "agreement",
        "sarcasm",
    ]

    df_subset = df_comments.iloc[:n_entries]

    def simple_progress(done, total, s):
        """
        Simple progress callback for logging.
        Prints status every 500 processed rows and at the end.
        """
        if done % 500 == 0 or done == total:
            print(
                f"[{done}/{total}] "
                f"written={s['written_records']} "
                f"skipped={s['skipped_records']} "
                f"errors={s['errors']}"
            )

    stats = label_dataframe(
        df=df_subset,
        tasks=tasks,
        ndjson_path=f"labels_{n_entries}.ndjson",
        batch_size=1000,
        skip_existing=True,
        progress=simple_progress,
        parent_col="parent_id",
        id_col="comment_id",
        text_col="body",
    )

    print(f"Labeling finished ({n_entries} entries).")
    print(stats)
else:
    print("df_comments is empty")

APIConnectionError: Connection error.

In [56]:
df_labels = pd.read_json(f"labels_{n_entries}.ndjson", lines=True)
df_labels

,comment_id,body,parent_id,comment_index,task,result,arguments,arguments_confidence,arguments_error,meta
0,la7kwyu,\nRemember that TrueReddit is a place to engag...,1do7blq,0,stance_intensity,"{'task': 'stance_intensity', 'score': 'ABSTAIN...",[],1.0,None,"{'thread_id': '1do7blq', 'submission_id': '1do..."
1,la7kwyu,\nRemember that TrueReddit is a place to engag...,1do7blq,0,civility,"{'task': 'civility', 'label': 6, 'confidence':...",[],1.0,None,"{'thread_id': '1do7blq', 'submission_id': '1do..."
2,la7kwyu,\nRemember that TrueReddit is a place to engag...,1do7blq,0,epistemic_modality,"{'task': 'epistemic_modality', 'score': 'ABSTA...",[],1.0,None,"{'thread_id': '1do7blq', 'submission_id': '1do..."
3,la7kwyu,\nRemember that TrueReddit is a place to engag...,1do7blq,0,justification_density,"{'task': 'justification_density', 'score': 'AB...",[],1.0,None,"{'thread_id': '1do7blq', 'submission_id': '1do..."
4,la7kwyu,\nRemember that TrueReddit is a place to engag...,1do7blq,0,responsiveness,"{'task': 'responsiveness', 'score': 'ABSTAIN',...",[],1.0,None,"{'thread_id': '1do7blq', 'submission_id': '1do..."
...,...,...,...,...,...,...,...,...,...,...
2795,l8grj5c,Tired of reporting this thread? [Debate us on...,1dcwjon,399,epistemic_modality,"{'task': 'epistemic_modality', 'score': 'ABSTA...",[],1.0,None,"{'thread_id': '1dcwjon', 'submission_id': '1dc..."
2796,l8grj5c,Tired of reporting this thread? [Debate us on...,1dcwjon,399,justification_density,"{'task': 'justification_density', 'score': 'AB...",[],1.0,None,"{'thread_id': '1dcwjon', 'submission_id': '1dc..."
2797,l8grj5c,Tired of reporting this thread? [Debate us on...,1dcwjon,399,responsiveness,"{'task': 'responsiveness', 'score': 'ABSTAIN',...",[],1.0,None,"{'thread_id': '1dcwjon', 'submission_id': '1dc..."
2798,l8grj5c,Tired of reporting this thread? [Debate us on...,1dcwjon,399,agreement,"{'task': 'agreement', 'score': 'ABSTAIN', 'con...",[],1.0,None,"{'thread_id': '1dcwjon', 'submission_id': '1dc..."
